# Session 3: N assets, the frontier, and the limits of diversification
**A. Real-world problem.** Session 2 mixed two assets. A real portfolio may hold dozens.
Three questions follow. (1) If I add more and more assets, does the risk keep falling to
zero? (2) With several assets, which mixes give the least risk for a chosen return, and which
mix has the best return per unit of risk? (3) Those answers are computed from numbers
*estimated* from a short history. How much should we trust them?

**Safety note.** Everything here is synthetic. The four assets, their returns and their
correlations are made up. This is NOT investment advice, and models like this one do not
predict real markets: real returns have fat tails, correlations change, and past performance
does not predict the future.

**Plan.** (1) N equal-weight assets with a common correlation, (2) the covariance matrix and
the efficient frontier for 4 assets, (3) the Sharpe ratio and the tangency portfolio,
(4) estimation error, (5) two failure cases.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: N equal-weight assets with a common correlation
Every asset has standard deviation s and every pair has correlation rho. With weights 1/N,
the variance of the portfolio is

    var = s^2 * (1/N + (1 - 1/N) * rho)

The first part, s^2/N, is the *idiosyncratic* risk (each asset's own wobble): it shrinks as N
grows. The second part tends to s^2 * rho, the *market* risk shared by all: no number of
assets removes it. We check the formula with a covariance matrix and with a simulation in
which every asset = a shared market part + its own part.

In [ ]:
def equal_weight_var(n, s, rho):
    return s ** 2 * (1.0 / n + (1.0 - 1.0 / n) * rho)


def equal_weight_var_matrix(n, s, rho):
    """Same quantity from w' Sigma w, with Sigma built entry by entry."""
    sigma = np.full((n, n), rho * s * s)
    np.fill_diagonal(sigma, s * s)
    w = np.full(n, 1.0 / n)
    return float(w @ sigma @ w)


def simulate_equal_weight_std(n, s, rho, years=200000, seed=5):
    """Std of the equal-weight portfolio of n simulated assets: asset = s*(sqrt(rho)*market + sqrt(1-rho)*own)."""
    rng = np.random.default_rng(seed)
    market = rng.standard_normal(years)
    own = rng.standard_normal((years, n))
    assets = s * (np.sqrt(rho) * market[:, None] + np.sqrt(1 - rho) * own)
    return float(np.std(assets.mean(axis=1)))


def market_floor(s, rho):
    """The risk that remains however many assets we add: s * sqrt(rho)."""
    return s * np.sqrt(rho)

## Step 2: four made-up assets and their covariance matrix
With several assets we keep the means in a list `mu`, the standard deviations in `sd` and the
correlations in a table `CORR`. The covariance matrix is Sigma[i][j] = corr[i][j] sd[i] sd[j].
A covariance matrix must be *positive definite* (every portfolio has positive variance); we
check that all its eigenvalues are above zero.

In [ ]:
NAMES = ["A", "B", "C", "D"]
MU = np.array([0.06, 0.08, 0.10, 0.12])
SD = np.array([0.10, 0.15, 0.20, 0.25])
CORR = np.array([[1.0, 0.3, 0.2, 0.1],
                 [0.3, 1.0, 0.4, 0.3],
                 [0.2, 0.4, 1.0, 0.5],
                 [0.1, 0.3, 0.5, 1.0]])
RF = 0.02                 # a made-up risk-free rate for the Sharpe ratio


def covariance(sd, corr):
    return corr * np.outer(sd, sd)


SIGMA = covariance(SD, CORR)


def is_positive_definite(sigma):
    return bool(np.all(np.linalg.eigvalsh(sigma) > 1e-12))


def port_stats(w, mu, sigma):
    """Expected return and risk (std) of weights w."""
    return float(w @ mu), float(np.sqrt(w @ sigma @ w))

## Step 3: the efficient frontier
For a target return m, which weights (summing to 1) give the smallest variance? Lagrange
multipliers give a closed form. With inv = Sigma^-1 and a vector of ones `one`:

    A = one' inv one,  B = one' inv mu,  C = mu' inv mu,  D = A C - B^2
    w(m) = inv (lam one + gam mu),  lam = (C - B m)/D,  gam = (A m - B)/D
    var(m) = (A m^2 - 2 B m + C) / D

The weights here may be negative (short selling) - that is what "closed form" allows.
The least-risk portfolio of all (any return) is w = inv one / A.

In [ ]:
def frontier_constants(mu, sigma):
    inv = np.linalg.inv(sigma)
    one = np.ones(len(mu))
    a = float(one @ inv @ one)
    b = float(one @ inv @ mu)
    c = float(mu @ inv @ mu)
    return inv, a, b, c, a * c - b * b


def frontier_weights(m, mu, sigma):
    inv, a, b, c, d = frontier_constants(mu, sigma)
    one = np.ones(len(mu))
    lam, gam = (c - b * m) / d, (a * m - b) / d
    return inv @ (lam * one + gam * mu)


def frontier_var(m, mu, sigma):
    _, a, b, c, d = frontier_constants(mu, sigma)
    return (a * m * m - 2 * b * m + c) / d


def min_variance_weights(sigma):
    inv = np.linalg.inv(sigma)
    x = inv @ np.ones(len(sigma))
    return x / x.sum()


def random_portfolios(n, n_assets=4, seed=3):
    """Random long-only weights (each row sums to 1)."""
    return np.random.default_rng(seed).dirichlet(np.ones(n_assets), size=n)

## Step 4: Sharpe ratio and the tangency portfolio
The Sharpe ratio is the extra return above the risk-free rate per unit of risk:
(return - rf) / risk. The portfolio with the highest Sharpe ratio (the *tangency* portfolio)
has weights proportional to inv (mu - rf one), scaled to sum to 1. Its Sharpe ratio is
sqrt((mu - rf)' inv (mu - rf)).

In [ ]:
def sharpe(w, mu, sigma, rf=RF):
    r, s = port_stats(w, mu, sigma)
    return (r - rf) / s


def tangency_weights(mu, sigma, rf=RF):
    x = np.linalg.solve(sigma, mu - rf)
    return x / x.sum()


def max_sharpe(mu, sigma, rf=RF):
    ex = mu - rf
    return float(np.sqrt(ex @ np.linalg.solve(sigma, ex)))

## Step 5: estimation error
In real life nobody knows mu and Sigma. We estimate them from T past periods, then build the
"optimal" portfolio from the estimates. We can test this honestly only in a simulation where
we *do* know the truth: draw T periods from the true model, estimate, optimise, then measure
the portfolio's risk under the true Sigma (out-of-sample) and compare with the risk the
estimate predicted (in-sample).

In [ ]:
def draw_sample(t, mu, sigma, seed):
    """t periods of correlated returns: mu + z L', where L L' = Sigma (Cholesky) and z is standard normal."""
    rng = np.random.default_rng(seed)
    return mu + rng.standard_normal((t, len(mu))) @ np.linalg.cholesky(sigma).T


def estimation_experiment(t, reps, mu=MU, sigma=SIGMA, seed=11):
    """Average over `reps` samples of length t: predicted risk, true risk and true Sharpe
    of the estimated minimum-variance and maximum-Sharpe portfolios."""
    rng = np.random.default_rng(seed)
    chol = np.linalg.cholesky(sigma).T
    pred, true, tsh, psh = [], [], [], []
    for _ in range(reps):
        x = mu + rng.standard_normal((t, len(mu))) @ chol
        mu_hat, sig_hat = x.mean(axis=0), np.cov(x.T)
        w = min_variance_weights(sig_hat)
        pred.append(np.sqrt(w @ sig_hat @ w))
        true.append(np.sqrt(w @ sigma @ w))
        v = tangency_weights(mu_hat, sig_hat)
        psh.append(sharpe(v, mu_hat, sig_hat))
        tsh.append(sharpe(v, mu, sigma))
    return {"pred_risk": float(np.mean(pred)), "true_risk": float(np.mean(true)),
            "pred_sharpe": float(np.mean(psh)), "true_sharpe": float(np.mean(tsh))}

## Step 6: two failure cases
1. **Correlations rise in a crisis.** The same N assets with a higher rho have a higher
   floor s * sqrt(rho): the benefit shrinks just when it is needed.
2. **Near-singular covariance.** If one asset is almost a copy of another, `inv` becomes
   huge, and the "optimal" weights explode into big long and short positions that nobody
   should trust. `near_clone_sigma` makes asset D follow asset C closely with a small
   independent part `eps`. We then nudge D's mean return by half a percentage point and
   watch the tangency weights jump.

In [ ]:
def near_clone_sigma(eps, sigma=SIGMA, sd=SD):
    """Covariance where asset D = (sd_D / sd_C) * asset C plus an independent part of std eps."""
    s = sigma.copy()
    k = sd[3] / sd[2]
    s[3, :3] = k * s[2, :3]
    s[:3, 3] = s[3, :3]
    s[3, 3] = k * k * s[2, 2] + eps ** 2
    return s


def condition_number(sigma):
    return float(np.linalg.cond(sigma))


def nudged_mu(delta=0.005):
    """The same mean returns, but asset D's mean raised by delta (0.005 = half a percentage point):
    the kind of small change that estimation error causes."""
    return MU + np.array([0.0, 0.0, 0.0, delta])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    s_, rho_ = 0.20, 0.3
    print("N equal-weight, s=0.20, rho=0.3:")
    for n in (1, 2, 4, 10, 100, 1000):
        print(f"  N={n:5d}: std formula {np.sqrt(equal_weight_var(n, s_, rho_)):.4f}")
    for n in (1, 2, 4, 10):
        print(f"  N={n:2d}: matrix {np.sqrt(equal_weight_var_matrix(n, s_, rho_)):.4f}  "
              f"simulated {simulate_equal_weight_std(n, s_, rho_):.4f}")
    print(f"  floor s*sqrt(rho) = {market_floor(s_, rho_):.4f}; N=10 at rho 0.8: "
          f"{np.sqrt(equal_weight_var(10, s_, 0.8)):.4f}")
    print("eigenvalues of Sigma:", np.round(np.linalg.eigvalsh(SIGMA), 5), "PD:", is_positive_definite(SIGMA))

    wmv = min_variance_weights(SIGMA)
    print("min-variance weights", np.round(wmv, 4), "ret/risk", np.round(port_stats(wmv, MU, SIGMA), 4))
    wt = tangency_weights(MU, SIGMA)
    print("tangency weights", np.round(wt, 4), "ret/risk", np.round(port_stats(wt, MU, SIGMA), 4),
          "sharpe", round(sharpe(wt, MU, SIGMA), 4), "max_sharpe", round(max_sharpe(MU, SIGMA), 4))
    for m in (0.08, 0.10):
        w = frontier_weights(m, MU, SIGMA)
        print(f"frontier at m={m}: weights {np.round(w, 4)} sum {w.sum():.6f} risk {np.sqrt(frontier_var(m, MU, SIGMA)):.4f}")
    rp = random_portfolios(5000)
    rr = rp @ MU
    rs = np.sqrt(np.einsum("ij,jk,ik->i", rp, SIGMA, rp))
    gap = rs - np.sqrt(frontier_var(rr, MU, SIGMA))
    print("random portfolios: min gap to frontier", gap.min(), "best sharpe", ((rr - RF) / rs).max())

    for t in (12, 24, 60, 240):
        e = estimation_experiment(t, 1000)
        print(f"T={t}: predicted risk {e['pred_risk']:.4f}, true risk {e['true_risk']:.4f}, "
              f"min possible {np.sqrt(wmv @ SIGMA @ wmv):.4f}; "
              f"Sharpe predicted {e['pred_sharpe']:.3f} true {e['true_sharpe']:.3f} best {max_sharpe(MU, SIGMA):.3f}")
    for eps in (0.1, 0.05, 0.01, 0.001):
        sg = near_clone_sigma(eps)
        print(f"near clone eps={eps}: cond {condition_number(sg):.3g} tangency {np.round(tangency_weights(MU, sg), 2)} "
              f"nudged {np.round(tangency_weights(nudged_mu(), sg), 2)}")

    # figures
    ns = np.arange(1, 101)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    for rho, c in [(0.1, "#2a9d8f"), (0.3, "#e0a030"), (0.8, "#d9534f")]:
        ax.plot(ns, 100 * np.sqrt(equal_weight_var(ns, s_, rho)), color=c, label=f"rho = {rho}")
        ax.axhline(100 * market_floor(s_, rho), color=c, ls=":")
    ax.plot([1, 2, 5, 10, 25, 50], [100 * simulate_equal_weight_std(n, s_, 0.3) for n in (1, 2, 5, 10, 25, 50)],
            "ko", ms=4, label="simulation (rho = 0.3)")
    ax.set_xlabel("number of assets N (equal weights)")
    ax.set_ylabel("portfolio risk, std (%)")
    ax.set_title("Risk falls with N, but stops at the market floor (dotted)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "diversification_vs_n.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.4))
    ax2.scatter(100 * rs, 100 * rr, s=4, alpha=0.35, color="#888888", label="5000 random long-only mixes")
    ms = np.linspace(0.04, 0.16, 100)
    ax2.plot(100 * np.sqrt(frontier_var(ms, MU, SIGMA)), 100 * ms, color="#2a6fdb", lw=2, label="efficient frontier")
    ax2.plot(100 * SD, 100 * MU, "ks")
    for nm, sd_, mu_ in zip(NAMES, SD, MU):
        ax2.annotate(nm, (100 * sd_, 100 * mu_), textcoords="offset points", xytext=(6, -4))
    ax2.plot(*(100 * np.array(port_stats(wt, MU, SIGMA))[::-1]), "r*", ms=12, label="max Sharpe")
    ax2.plot(*(100 * np.array(port_stats(wmv, MU, SIGMA))[::-1]), "g^", ms=8, label="min variance")
    ax2.set_xlabel("risk, std of yearly return (%)")
    ax2.set_ylabel("expected yearly return (%)")
    ax2.set_title("No random mix beats the frontier")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "efficient_frontier.png", dpi=120)

    ts = [12, 24, 36, 60, 120, 240]
    res = [estimation_experiment(t, 600) for t in ts]
    fig3, ax3 = plt.subplots(figsize=(6.4, 4.2))
    ax3.plot(ts, [100 * r_["pred_risk"] for r_ in res], "o-", label="risk the estimate predicted (in-sample)")
    ax3.plot(ts, [100 * r_["true_risk"] for r_ in res], "s-", label="risk the portfolio really has (out-of-sample)")
    ax3.axhline(100 * np.sqrt(wmv @ SIGMA @ wmv), color="gray", ls=":", label="best possible (true minimum)")
    ax3.set_xscale("log")
    ax3.set_xlabel("length of the history T (periods)")
    ax3.set_ylabel("portfolio risk, std (%)")
    ax3.set_title("A short history makes the portfolio look safer than it is")
    ax3.grid(alpha=0.3)
    ax3.legend(fontsize=8)
    fig3.tight_layout()
    fig3.savefig(OUT / "estimation_error.png", dpi=120)

## Where this stops
The model assumes bell-curve returns, fixed correlations and known costs of nothing. It
ignores trading costs, taxes, and the fact that real correlations jump in a crisis. Better
estimates (shrinkage), weight limits and robust methods are the road to the Target: see
`promo/roadmap.md`.

## Coding tasks
1. **Easy.** Change `CORR` so that all correlations are 0.1 and print the minimum-variance weights and risk. Predict whether the risk goes up or down first.
2. **Stretch.** Write `frontier_table()` that prints the frontier risk for target returns 0.07, 0.08, ..., 0.12 and check that none of them is below the minimum-variance risk.
3. **Explain.** Explain, in your own words, why the risk the estimate predicts is lower than the risk the portfolio really has.

## Assessment
- **A. Conceptual.** Why can you never diversify away all the risk by adding more and more assets, if they share a common correlation?
- **B. Numerical.** Find the variance and risk of an equal-weight portfolio of N = 10 assets with s = 0.20 and rho = 0.3. By what percent is it below the risk of one asset?
- **C. Graph and simulation interpretation.** In the estimation-error figure, what is the predicted risk and the true risk for a history of T = 12 periods, and for T = 240? What does the gap tell you?
- **D. Design and optimisation challenge.** Design: with s = 0.20 and rho = 0.3 you want a risk of at most 12 percent using equal weights. How many assets do you need? What if the assets had rho = 0.5? What does that say about a crisis?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # ---- N equal-weight assets, common correlation
    s = 0.20
    ok = all(abs(L.equal_weight_var(n, s, q) - L.equal_weight_var_matrix(n, s, q)) < 1e-12
             for n in (1, 2, 4, 10, 50) for q in (0.0, 0.3, 0.8, 1.0))
    checks.append(("N equal weights: the formula s^2 (1/N + (1 - 1/N) rho) equals w' Sigma w from a matrix", ok))
    sim_ok = all(abs(L.simulate_equal_weight_std(n, s, 0.3) - np.sqrt(L.equal_weight_var(n, s, 0.3))) < 0.001 for n in (1, 2, 4, 10))
    checks.append(("a seeded simulation (200000 years) agrees with the formula within 0.001 for N = 1, 2, 4, 10", sim_ok))
    checks.append(("the simulated N = 4 portfolio (seed 5, 200000 years) has risk 0.1380",
                   abs(L.simulate_equal_weight_std(4, s, 0.3) - 0.1380) < 5e-5))
    # worked example: N = 4, s = 0.2, rho = 0.3: variance 0.019, std 0.1378
    checks.append(("worked example: N = 4, s = 0.20, rho = 0.3 gives variance 0.0190 and std 0.1378 (31% below one asset)",
                   abs(L.equal_weight_var(4, 0.2, 0.3) - 0.04 * (0.25 + 0.75 * 0.3)) < 1e-15
                   and abs(L.equal_weight_var(4, 0.2, 0.3) - 0.019) < 1e-12
                   and abs(np.sqrt(L.equal_weight_var(4, 0.2, 0.3)) - 0.1378) < 5e-5
                   and abs(1 - np.sqrt(L.equal_weight_var(4, 0.2, 0.3)) / 0.2 - 0.31) < 0.005))
    checks.append(("the risk falls with N: 0.2000 (N=1), 0.1612 (2), 0.1217 (10), 0.1108 (100) at rho = 0.3",
                   all(abs(np.sqrt(L.equal_weight_var(n, s, 0.3)) - v) < 5e-5 for n, v in ((1, 0.2), (2, 0.1612), (10, 0.1217), (100, 0.1108)))))
    floor = L.market_floor(s, 0.3)
    checks.append(("the market floor s sqrt(rho) is 0.1095, and a million assets get within 1e-4 of it, never below",
                   abs(floor - 0.1095) < 5e-5 and 0 <= np.sqrt(L.equal_weight_var(10 ** 6, s, 0.3)) - floor < 1e-4
                   and all(np.sqrt(L.equal_weight_var(n, s, 0.3)) > floor for n in (1, 10, 100, 1000))))
    checks.append(("edge: rho = 0 gives s^2/N (risk vanishes as 1/sqrt(N)); rho = 1 gives s^2 for every N (no benefit)",
                   all(abs(L.equal_weight_var(n, s, 0.0) - s * s / n) < 1e-15 and abs(L.equal_weight_var(n, s, 1.0) - s * s) < 1e-15 for n in (1, 5, 50))))
    r3, r8 = np.sqrt(L.equal_weight_var(10, s, 0.3)), np.sqrt(L.equal_weight_var(10, s, 0.8))
    checks.append(("failure case: with N = 10, raising rho from 0.3 to 0.8 raises the risk from 0.1217 to 0.1811 and cuts the benefit from 39% to 9%",
                   abs(r3 - 0.1217) < 5e-5 and abs(r8 - 0.1811) < 5e-5
                   and abs((1 - r3 / s) - 0.39) < 0.005 and abs((1 - r8 / s) - 0.09) < 0.005))

    checks.append(("hand calculation: N = 10, s = 0.20, rho = 0.3 gives variance 0.0148 (0.04 * 0.37)",
                   abs(L.equal_weight_var(10, 0.2, 0.3) - 0.0148) < 1e-12))
    n_need = next(n for n in range(1, 200) if np.sqrt(L.equal_weight_var(n, 0.2, 0.3)) <= 0.12)
    checks.append(("design: risk <= 12% at rho = 0.3 needs N = 12 (0.1197; N = 11 gives 0.1206), and rho = 0.5 is impossible (floor 0.1414)",
                   n_need == 12 and abs(np.sqrt(L.equal_weight_var(12, 0.2, 0.3)) - 0.1197) < 5e-5
                   and abs(np.sqrt(L.equal_weight_var(11, 0.2, 0.3)) - 0.1206) < 5e-5 and abs(L.market_floor(0.2, 0.5) - 0.1414) < 5e-5))

    # ---- covariance matrix
    sig = L.SIGMA
    ev = np.linalg.eigvalsh(sig)
    checks.append(("Sigma is symmetric and positive definite (smallest eigenvalue 0.00851)",
                   np.allclose(sig, sig.T) and L.is_positive_definite(sig) and abs(ev.min() - 0.00851) < 5e-6))
    np.linalg.cholesky(sig)
    bad = np.array([[1, 0.9, -0.9], [0.9, 1, 0.9], [-0.9, 0.9, 1.0]])
    checks.append(("edge: an impossible correlation table (0.9, 0.9, -0.9) is rejected as not positive definite",
                   not L.is_positive_definite(bad) and L.is_positive_definite(np.eye(3))))
    checks.append(("Sigma diagonal equals sd^2 and the correlations are recovered",
                   np.allclose(np.diag(sig), L.SD ** 2) and np.allclose(sig / np.outer(L.SD, L.SD), L.CORR)))

    # ---- frontier
    wmv = L.min_variance_weights(sig)
    rmv, smv = L.port_stats(wmv, L.MU, sig)
    checks.append(("minimum-variance portfolio: weights 0.7232, 0.1723, 0.0515, 0.0530; return 0.0687; risk 0.0914; weights sum to 1",
                   np.allclose(wmv, [0.7232, 0.1723, 0.0515, 0.0530], atol=5e-5) and abs(wmv.sum() - 1) < 1e-12
                   and abs(rmv - 0.0687) < 5e-5 and abs(smv - 0.0914) < 5e-5))
    ok = True
    for m in (0.07, 0.08, 0.10, 0.12):
        w = L.frontier_weights(m, L.MU, sig)
        ok &= abs(w.sum() - 1) < 1e-12 and abs(w @ L.MU - m) < 1e-12 and abs(w @ sig @ w - L.frontier_var(m, L.MU, sig)) < 1e-12
    checks.append(("frontier weights sum to 1, hit the target return, and have the variance given by the closed form", ok))
    w8 = L.frontier_weights(0.08, L.MU, sig)
    w10 = L.frontier_weights(0.10, L.MU, sig)
    checks.append(("frontier at 8% return: weights 0.4832, 0.2087, 0.1329, 0.1752 and risk 0.1022; at 10%: risk 0.1561",
                   np.allclose(w8, [0.4832, 0.2087, 0.1329, 0.1752], atol=5e-5)
                   and abs(np.sqrt(L.frontier_var(0.08, L.MU, sig)) - 0.1022) < 5e-5
                   and abs(np.sqrt(L.frontier_var(0.10, L.MU, sig)) - 0.1561) < 5e-5 and w10.min() > 0))
    b = (np.ones(4) @ np.linalg.solve(sig, L.MU)) / (np.ones(4) @ np.linalg.solve(sig, np.ones(4)))
    checks.append(("the frontier at the minimum-variance return B/A is the minimum-variance portfolio",
                   np.allclose(L.frontier_weights(b, L.MU, sig), wmv, atol=1e-10)))
    rp = L.random_portfolios(5000)
    rr = rp @ L.MU
    rs = np.sqrt(np.einsum("ij,jk,ik->i", rp, sig, rp))
    gap = rs - np.sqrt(L.frontier_var(rr, L.MU, sig))
    checks.append(("5000 random long-only portfolios: none has less risk than the frontier at its own return",
                   bool(np.all(rp >= 0)) and np.allclose(rp.sum(axis=1), 1) and gap.min() > -1e-12))
    rng = np.random.default_rng(0)
    ok = True
    for _ in range(200):
        d = rng.standard_normal(4)
        d -= np.ones(4) * d.mean()                          # keep the weights summing to 1
        w = w8 + 0.05 * d
        ok &= w @ sig @ w >= L.frontier_var(w @ L.MU, L.MU, sig) - 1e-12
    checks.append(("random nudges of the frontier weights (weights summing to 1, any shorts) never beat the frontier at their own return", ok))

    # ---- Sharpe and tangency
    wt = L.tangency_weights(L.MU, sig)
    checks.append(("tangency portfolio: weights 0.5006, 0.2061, 0.1270, 0.1664; return 0.0792; risk 0.1007; Sharpe 0.5876",
                   np.allclose(wt, [0.5006, 0.2061, 0.1270, 0.1664], atol=5e-5) and abs(wt.sum() - 1) < 1e-12
                   and abs(L.port_stats(wt, L.MU, sig)[0] - 0.0792) < 5e-5 and abs(L.port_stats(wt, L.MU, sig)[1] - 0.1007) < 5e-5
                   and abs(L.sharpe(wt, L.MU, sig) - 0.5876) < 5e-5))
    checks.append(("the closed-form maximum Sharpe ratio sqrt((mu-rf)' inv (mu-rf)) equals the tangency portfolio's Sharpe",
                   abs(L.max_sharpe(L.MU, sig) - L.sharpe(wt, L.MU, sig)) < 1e-12))
    best_random = float(((rr - L.RF) / rs).max())
    checks.append(("no random portfolio beats the tangency Sharpe ratio (best random 0.5875 against 0.5876)",
                   best_random <= L.max_sharpe(L.MU, sig) + 1e-12 and abs(best_random - 0.5875) < 5e-5))
    checks.append(("each single asset has Sharpe ratio 0.4000, lower than the tangency mix (0.5876)",
                   all(abs(L.sharpe(np.eye(4)[i], L.MU, sig) - 0.4) < 1e-12 for i in range(4))
                   and 0.4 < L.max_sharpe(L.MU, sig)))

    stress = L.covariance(L.SD, np.where(np.eye(4) == 1, 1.0, 0.8))
    checks.append(("stress test (all correlations 0.8): the minimum-variance portfolio has risk 0.1144 (from 0.0914), the tangency portfolio 0.1367 (from 0.1007)",
                   abs(np.sqrt(wmv @ stress @ wmv) - 0.1144) < 5e-5 and abs(np.sqrt(wt @ stress @ wt) - 0.1367) < 5e-5
                   and L.is_positive_definite(stress)))
    contrib = wmv * (sig @ wmv) / (wmv @ sig @ wmv)
    checks.append(("risk contributions of the minimum-variance portfolio equal its weights: asset A carries 72% of the risk",
                   np.allclose(contrib, wmv, atol=1e-12) and abs(contrib[0] - 0.72) < 0.005))

    # ---- estimation error
    res = {t: L.estimation_experiment(t, 1000) for t in (12, 24, 60, 240)}
    checks.append(("estimation error T = 24: predicted risk 0.0847, true risk 0.0982, best possible 0.0914",
                   abs(res[24]["pred_risk"] - 0.0847) < 1e-4 and abs(res[24]["true_risk"] - 0.0982) < 1e-4 and abs(smv - 0.0914) < 5e-5))
    checks.append(("estimation error T = 12: predicted 0.0753 against true 0.1080; T = 60: 0.0887 against 0.0938; T = 240: 0.0906 against 0.0919",
                   abs(res[12]["pred_risk"] - 0.0753) < 1e-4 and abs(res[12]["true_risk"] - 0.1080) < 1e-4
                   and abs(res[60]["pred_risk"] - 0.0887) < 1e-4 and abs(res[60]["true_risk"] - 0.0938) < 1e-4
                   and abs(res[240]["pred_risk"] - 0.0906) < 1e-4 and abs(res[240]["true_risk"] - 0.0919) < 1e-4))
    checks.append(("out-of-sample risk is above the predicted risk and above the true minimum at every T, and the gaps shrink as T grows",
                   all(res[t]["true_risk"] > res[t]["pred_risk"] and res[t]["true_risk"] > smv for t in res)
                   and res[12]["true_risk"] - res[12]["pred_risk"] > res[24]["true_risk"] - res[24]["pred_risk"]
                   > res[60]["true_risk"] - res[60]["pred_risk"] > res[240]["true_risk"] - res[240]["pred_risk"]))
    checks.append(("Sharpe-best portfolio from T = 24: the estimate promises 0.760, the truth is 0.459, the best possible is 0.588",
                   abs(res[24]["pred_sharpe"] - 0.760) < 1e-3 and abs(res[24]["true_sharpe"] - 0.459) < 1e-3
                   and abs(L.max_sharpe(L.MU, sig) - 0.588) < 5e-4))
    x = L.draw_sample(200000, L.MU, sig, 1)
    checks.append(("draw_sample reproduces the target means and covariance (200000 periods, within 0.002)",
                   np.allclose(x.mean(axis=0), L.MU, atol=0.002) and np.allclose(np.cov(x.T), sig, atol=0.002)
                   and np.array_equal(L.draw_sample(10, L.MU, sig, 3), L.draw_sample(10, L.MU, sig, 3))))

    # ---- failure case: near-singular covariance
    s1, s2 = L.near_clone_sigma(0.1), L.near_clone_sigma(0.01)
    w1, w2 = L.tangency_weights(L.nudged_mu(), s1), L.tangency_weights(L.nudged_mu(), s2)
    checks.append(("near-clone assets stay positive definite, but the condition number grows from 31 (eps = 0.1) to 2756 (eps = 0.01)",
                   L.is_positive_definite(s1) and L.is_positive_definite(s2)
                   and abs(L.condition_number(s1) - 31.3) < 0.1 and abs(L.condition_number(s2) - 2756) < 5))
    checks.append(("failure case: nudging D's mean by 0.5 points moves the tangency weights of C and D from 0.12, 0.09 (eps = 0.1) to 8.81, -7.19 (eps = 0.01)",
                   np.allclose(w1[2:], [0.12, 0.09], atol=0.005) and np.allclose(w2[2:], [8.81, -7.19], atol=0.01)
                   and abs(w2.sum() - 1) < 1e-9 and abs(w2).max() > 5))
    checks.append(("without the nudge the tangency weights barely move with eps: 0.51, 0.25, 0.23, 0.00",
                   np.allclose(L.tangency_weights(L.MU, s2), [0.51, 0.25, 0.23, 0.0], atol=0.005)))

    # ---- mini-project data (the numbers printed in the README)
    mu = np.array([0.04, 0.07, 0.09, 0.13])
    sd = np.array([0.08, 0.12, 0.18, 0.30])
    corr = np.array([[1, 0.2, 0.1, 0.0], [0.2, 1, 0.3, 0.2], [0.1, 0.3, 1, 0.6], [0.0, 0.2, 0.6, 1.0]])
    sg = L.covariance(sd, corr)
    wb = L.min_variance_weights(sg)
    vb = L.tangency_weights(mu, sg)
    e = L.estimation_experiment(36, 1000, mu, sg, 21)
    checks.append(("mini-project (made-up set B): positive definite; minimum-variance weights 0.7027, 0.2174, 0.0650, 0.0148 with risk 0.0708",
                   L.is_positive_definite(sg) and np.allclose(wb, [0.7027, 0.2174, 0.0650, 0.0148], atol=5e-5)
                   and abs(L.port_stats(wb, mu, sg)[1] - 0.0708) < 5e-5))
    checks.append(("mini-project: Sharpe-best weights 0.3545, 0.3918, 0.1379, 0.1158 with Sharpe 0.5521",
                   np.allclose(vb, [0.3545, 0.3918, 0.1379, 0.1158], atol=5e-5) and abs(L.sharpe(vb, mu, sg) - 0.5521) < 5e-5))
    checks.append(("mini-project: with T = 36, 1000 samples (seed 21) the predicted risk averages 0.0669 and the true risk 0.0742",
                   abs(e["pred_risk"] - 0.0669) < 1e-4 and abs(e["true_risk"] - 0.0742) < 1e-4 and e["true_risk"] > e["pred_risk"]))
    xs = L.draw_sample(36, mu, sg, 21)
    wh = L.min_variance_weights(np.cov(xs.T))
    checks.append(("mini-project: the single sample with seed 21 gives weights 0.8165, 0.0106, 0.1897, -0.0168 (a short position in D)",
                   np.allclose(wh, [0.8165, 0.0106, 0.1897, -0.0168], atol=5e-4) and wh[3] < 0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
